# Autoencoder-based Clustering

#### Pipeline: DAE for cleaning -> Sparse AE for feature extraction -> UMAP-HBDSCAN for Cluster analysis

# Pipeline implement

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import umap
import hdbscan
from sklearn.metrics import silhouette_score
import cv2

# =====================
# Configuration Settings
# =====================
CONFIG = {
    "data_path": "/Users/user/Desktop/NonFKwork/AI/clusters/clusters/Sudah_mentok/cropped_cells/Myeloid",
    "image_size": (128, 128),
    "batch_size": 32,
    "denoiser_epochs": 50,
    "sparse_ae_epochs": 100,
    "latent_dim": 64,  # Dimension of latent space
    "sparsity_target": 0.05,  # Target activation sparsity (5%)
    "sparsity_weight": 0.5,  # Weight for sparsity loss
    "umap_n_neighbors": 15,
    "umap_min_dist": 0.1,
    "hdbscan_min_cluster_size": 10,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}
print(f"Using device: {CONFIG['device']}")


# =====================
# Data Loading & Preparation
# =====================
class CellDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.image_files = [f for f in os.listdir(root_dir)
                            if f.endswith(('.png', '.jpg', '.jpeg'))]
        self.transform = transforms.Compose([
            transforms.Resize(CONFIG['image_size']),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_files[idx])
        image = Image.open(img_path).convert('RGB')

        # Add synthetic noise for denoising training
        clean_image = self.transform(image)
        noisy_image = clean_image + 0.2 * torch.randn_like(clean_image)

        return noisy_image, clean_image


# Initialize dataset and dataloader
dataset = CellDataset(CONFIG['data_path'])
dataloader = DataLoader(dataset, batch_size=CONFIG['batch_size'], shuffle=True)


# =====================
# Model Architectures
# =====================
class DenoisingAE(nn.Module):
    """Denoising Autoencoder with skip connections"""

    def __init__(self):
        super(DenoisingAE, self).__init__()

        # Encoder
        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32))
        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64))

        # Decoder
        self.dec1 = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32))
        self.dec2 = nn.Sequential(
            nn.ConvTranspose2d(32, 3, 3, stride=2, padding=1, output_padding=1),
            nn.Tanh())

    def forward(self, x):
        # Encoder
        x1 = self.enc1(x)
        x2 = self.enc2(x1)

        # Decoder with skip connections
        d1 = self.dec1(x2) + x1  # Skip connection
        d2 = self.dec2(d1)
        return d2


class SparseAutoencoder(nn.Module):
    """Sparse Autoencoder with KL-divergence sparsity constraint"""

    def __init__(self, latent_dim):
        super(SparseAutoencoder, self).__init__()
        self.latent_dim = latent_dim

        # Encoder
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * 32 * 32, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim)
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 64 * 32 * 32),
            nn.ReLU(),
            nn.Unflatten(1, (64, 32, 32)),
            nn.ConvTranspose2d(64, 32, 3, stride=2, padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 3, stride=2, padding=1, output_padding=1),
            nn.Tanh()
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return latent, reconstructed


# =====================
# Training Functions
# =====================
def train_denoiser():
    """Train denoising autoencoder"""
    model = DenoisingAE().to(CONFIG['device'])
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    print("Training Denoising Autoencoder...")
    for epoch in range(CONFIG['denoiser_epochs']):
        for noisy_imgs, clean_imgs in dataloader:
            noisy_imgs = noisy_imgs.to(CONFIG['device'])
            clean_imgs = clean_imgs.to(CONFIG['device'])

            # Forward pass
            outputs = model(noisy_imgs)
            loss = criterion(outputs, clean_imgs)

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch + 1}/{CONFIG['denoiser_epochs']}], Loss: {loss.item():.4f}")

    # Save model and return
    torch.save(model.state_dict(), "denoiser.pth")
    return model


def kl_divergence(p, q):
    """KL divergence for sparsity constraint"""
    p = torch.clamp(p, 1e-10, 1.0)
    q = torch.clamp(q, 1e-10, 1.0)
    return torch.sum(p * torch.log(p / q))


def train_sparse_ae(denoiser):
    """Train sparse autoencoder on denoised images"""
    model = SparseAutoencoder(CONFIG['latent_dim']).to(CONFIG['device'])
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0005)

    print("\nTraining Sparse Autoencoder...")
    for epoch in range(CONFIG['sparse_ae_epochs']):
        total_loss = 0.0
        for noisy_imgs, _ in dataloader:
            noisy_imgs = noisy_imgs.to(CONFIG['device'])

            # Denoise first
            with torch.no_grad():
                clean_imgs = denoiser(noisy_imgs)

            # Sparse AE forward
            latent, reconstructed = model(clean_imgs)

            # Calculate losses
            recon_loss = criterion(reconstructed, clean_imgs)

            # Sparsity constraint (KL divergence)
            avg_activation = torch.mean(torch.sigmoid(latent), dim=0)
            sparsity_loss = kl_divergence(
                avg_activation,
                torch.tensor([CONFIG['sparsity_target']], device=CONFIG['device'])
            )

            # Combined loss
            loss = recon_loss + CONFIG['sparsity_weight'] * sparsity_loss

            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        if (epoch + 1) % 10 == 0:
            avg_loss = total_loss / len(dataloader)
            print(f"Epoch [{epoch + 1}/{CONFIG['sparse_ae_epochs']}], Loss: {avg_loss:.4f}")

    # Save model and return
    torch.save(model.state_dict(), "sparse_ae.pth")
    return model


# =====================
# Embedding Extraction
# =====================
def extract_embeddings(denoiser, sparse_ae):
    """Extract latent embeddings for all images"""
    sparse_ae.eval()
    denoiser.eval()

    all_embeddings = []
    all_images = []
    filenames = []

    with torch.no_grad():
        for noisy_imgs, _ in dataloader:
            noisy_imgs = noisy_imgs.to(CONFIG['device'])

            # Denoise and get embeddings
            clean_imgs = denoiser(noisy_imgs)
            embeddings, _ = sparse_ae(clean_imgs)

            # Store results
            all_embeddings.append(embeddings.cpu().numpy())
            all_images.append(clean_imgs.cpu().numpy())
            filenames.extend(dataset.image_files[:len(noisy_imgs)])

    # Concatenate results
    embeddings = np.concatenate(all_embeddings, axis=0)
    images = np.concatenate(all_images, axis=0)

    return embeddings, images, filenames


# =====================
# Clustering & Visualization
# =====================
def visualize_clusters(embeddings_2d, cluster_labels, images, filenames):
    """Visualize UMAP clusters with sample images"""
    plt.figure(figsize=(15, 12))

    # Create scatter plot
    scatter = plt.scatter(
        embeddings_2d[:, 0], embeddings_2d[:, 1],
        c=cluster_labels, cmap='tab20', alpha=0.6, s=15
    )

    # Add sample images for each cluster
    unique_clusters = np.unique(cluster_labels)
    for cluster_id in unique_clusters:
        if cluster_id == -1:  # Skip noise points
            continue

        # Find cluster center
        cluster_points = embeddings_2d[cluster_labels == cluster_id]
        center = np.mean(cluster_points, axis=0)

        # Find closest point to center
        distances = np.linalg.norm(cluster_points - center, axis=1)
        closest_idx = np.argmin(distances)

        # Find original image index
        global_idx = np.where(cluster_labels == cluster_id)[0][closest_idx]
        sample_img = images[global_idx].transpose(1, 2, 0)
        sample_img = np.clip((sample_img * 0.5 + 0.5) * 255, 0, 255).astype(np.uint8)

        # Display as inset
        ax_inset = plt.axes([0.8, 0.1 + cluster_id * 0.05, 0.1, 0.1])
        ax_inset.imshow(sample_img)
        ax_inset.axis('off')
        ax_inset.set_title(f"Cluster {cluster_id}")

    plt.colorbar(scatter, label='Cluster ID')
    plt.title("UMAP Projection of Myeloid Cell Embeddings")
    plt.xlabel("UMAP Dimension 1")
    plt.ylabel("UMAP Dimension 2")
    plt.savefig("cluster_visualization.png", dpi=300)
    plt.show()


def analyze_clusters(embeddings, cluster_labels):
    """Calculate cluster metrics and characteristics"""
    # Calculate silhouette score (excluding noise points)
    valid_mask = cluster_labels != -1
    if np.sum(valid_mask) > 1:
        score = silhouette_score(embeddings[valid_mask], cluster_labels[valid_mask])
        print(f"Silhouette Score: {score:.3f}")

    # Print cluster sizes
    unique, counts = np.unique(cluster_labels, return_counts=True)
    print("\nCluster Distribution:")
    for label, count in zip(unique, counts):
        if label == -1:
            print(f"  Noise points: {count}")
        else:
            print(f"  Cluster {label}: {count} cells")


# =====================
# Main Pipeline
# =====================
def main():
    # Step 1: Train denoising autoencoder
    denoiser = train_denoiser()

    # Step 2: Train sparse autoencoder
    sparse_ae = train_sparse_ae(denoiser)

    # Step 3: Extract embeddings
    embeddings, denoised_images, filenames = extract_embeddings(denoiser, sparse_ae)
    print(f"\nExtracted embeddings shape: {embeddings.shape}")

    # Step 4: Dimensionality reduction with UMAP
    print("Running UMAP dimensionality reduction...")
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=CONFIG['umap_n_neighbors'],
        min_dist=CONFIG['umap_min_dist'],
        random_state=42
    )
    embeddings_2d = reducer.fit_transform(embeddings)

    # Step 5: Clustering with HDBSCAN
    print("Performing HDBSCAN clustering...")
    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=CONFIG['hdbscan_min_cluster_size'],
        gen_min_span_tree=True
    )
    cluster_labels = clusterer.fit_predict(embeddings_2d)

    # Save results for further analysis
    np.savez("myeloid_clustering_results.npz",
             embeddings=embeddings,
             embeddings_2d=embeddings_2d,
             cluster_labels=cluster_labels,
             filenames=filenames)


if __name__ == "__main__":
    main()